# Phase Functional: Descriptive Analysis

## Overview
Descriptive analysis of ACDC-discovered circuits for the LSC (Literal Sequence Copying) task across 4 Pythia models and 5 frequency bands.

## Key Questions
1. How do circuit properties (size, accuracy, KL divergence) vary with input token frequency?
2. Do circuits generalize across frequency bands?
3. Is there asymmetric transfer between low-frequency and high-frequency circuits?
4. How do circuit properties scale with model size?

## Domains
- **Faithfulness (Sufficiency)**: Same-band circuit accuracy
- **Completeness (Necessity)**: Ablation accuracy
- **Minimality**: Circuit size as fraction of total edges
- **Generalization**: Cross-band transfer matrices
- **Asymmetry**: LF->HF vs HF->LF directional transfer
- **Scaling**: Trends across 70M / 160M / 410M / 1B parameters

## Hypotheses
- H1: Higher-frequency tokens produce more faithful circuits (higher base accuracy = easier task)
- H2: Circuits are frequency-specific (generalization gap > 0)
- H3: Asymmetric transfer exists between frequency extremes
- H4: Larger models yield more faithful, sparser circuits

## Notebook Structure
1. Setup & Data Loading
2. Dataset Overview
3. Base Model Analysis
4. Circuit Structure (Minimality)
5. Same-Band Performance (Faithfulness)
6. Cross-Band Generalization
7. Asymmetric Transfer
8. Ablation Analysis (Completeness)
9. Model Scaling
10. Aggregated Summary & Export

## Data Sources
- 60 metrics.json files (4 models x 5 bands x 3 draws)
- Pre-computed ACDC circuits with Pareto-optimized thresholds
- Location: `LSC_circuits/circuit_discovery/circuits/`

## 1. Setup & Data Loading

In [1]:
import sys
import json
import warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

NOTEBOOK_DIR = Path("LSC_circuit_analysis/01_Phase_Functional")
sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.constants import *
from utils.data_loading import build_all_dataframes
from utils.metrics import (
    compute_transfer_matrix,
    compute_generalization_gap,
    compute_asymmetric_transfer,
    compute_pairwise_asymmetry_matrix,
    compute_model_scaling_df,
)
from utils.plotting import (
    setup_plotting,
    save_figure,
    get_band_labels,
    plot_transfer_heatmap,
    plot_metric_heatmap,
    plot_boxplot_by_band,
    plot_scaling_panel,
)

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()
print(f"Analysis output: {ANALYSIS_DIR}")
print(f"Visualization output: {VIZ_DIR}")

Analysis output: LSC_circuit_analysis/01_Phase_Functional/outputs/analysis
Visualization output: LSC_circuit_analysis/01_Phase_Functional/outputs/viz


In [2]:
df_circuit, df_transfer, raw_metrics = build_all_dataframes()
print(f"Circuit DataFrame: {df_circuit.shape} (one row per model x band x draw)")
print(
    f"Transfer DataFrame: {df_transfer.shape} (one row per model x train_band x draw x test_band)"
)
print(f"Raw metrics loaded: {len(raw_metrics)}")

Loaded 75 / 75 metrics files
Circuit DataFrame: (75, 27) (one row per model x band x draw)
Transfer DataFrame: (375, 18) (one row per model x train_band x draw x test_band)
Raw metrics loaded: 75


## 2. Dataset Overview

In [3]:
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")
print(f"Total circuits: {len(df_circuit)}")
print(f"Necessity tests passed: {df_circuit['necessity_pass'].sum()}/{len(df_circuit)}")
print()
print("Thresholds per model (Pareto-optimized):")
for m in MODELS:
    print(f"  {m}: {MODEL_THRESHOLDS[m]}")
print()
print("Total edges per model:")
for m in MODELS:
    print(f"  {m}: {MODEL_TOTAL_EDGES[m]:,} ({MODEL_LAYERS[m]}L, {MODEL_HEADS[m]}H)")
print()
print("Draws per (model, band):")
print(df_circuit.groupby(["model", "band"], observed=True).size().unstack(fill_value=0))

DATASET OVERVIEW
Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']
Total circuits: 75
Necessity tests passed: 75/75

Thresholds per model (Pareto-optimized):
  pythia-70m: 0.00158
  pythia-160m: 0.000631
  pythia-410m: 0.000251
  pythia-1b: 0.00158
  pythia-1.4b: 0.000631

Total edges per model:
  pythia-70m: 1,324 (6L, 8H)
  pythia-160m: 11,467 (12L, 12H)
  pythia-410m: 80,581 (24L, 16H)
  pythia-1b: 10,009 (16L, 8H)
  pythia-1.4b: 80,581 (24L, 16H)

Draws per (model, band):
band         low  medium  high  very_high  control
model                                             
pythia-70m     3       3     3          3        3
pythia-160m    3       3     3          3        3
pythia-410m    3       3     3          3        3
pythia-1b      3       3     3          3        3
pythia-1.4b    3       3     3          3        3


## 3. Base Model Analysis

How does the full (unpruned) model perform on each frequency band?

In [4]:
base_summary = (
    df_circuit.groupby(["model", "band"], observed=True)
    .agg(
        base_acc_mean=("base_accuracy", "mean"),
        base_acc_std=("base_accuracy", "std"),
        base_top5_mean=("base_top5_accuracy", "mean"),
        base_top5_std=("base_top5_accuracy", "std"),
        base_mcp_mean=("base_mean_correct_prob", "mean"),
        base_mcp_std=("base_mean_correct_prob", "std"),
    )
    .reset_index()
)
print(base_summary.to_string(index=False))
base_summary.to_csv(ANALYSIS_DIR / "base_model_stats.csv", index=False)
print(f"\nSaved: base_model_stats.csv")

      model      band  base_acc_mean  base_acc_std  base_top5_mean  base_top5_std  base_mcp_mean  base_mcp_std
 pythia-70m       low       0.300741      0.053024        0.454815       0.065067       0.112005      0.020876
 pythia-70m    medium       0.400000      0.024746        0.557037       0.028574       0.163906      0.008848
 pythia-70m      high       0.502222      0.039503        0.647407       0.051127       0.202822      0.006882
 pythia-70m very_high       0.669630      0.032152        0.810370       0.017962       0.281973      0.019963
 pythia-70m   control       0.595556      0.035556        0.774815       0.028574       0.243850      0.017432
pythia-160m       low       0.928889      0.004444        0.991111       0.004444       0.414234      0.008471
pythia-160m    medium       0.957037      0.002566        0.994074       0.010264       0.457608      0.000748
pythia-160m      high       0.960000      0.008889        0.995556       0.004444       0.507066      0.001693
p

### VIZ 01: Base Model Accuracy by Band

In [5]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 6), sharey=True)
for i, model in enumerate(MODELS):
    ax = axes[i]
    mdata = df_circuit[df_circuit["model"] == model]
    sns.boxplot(
        data=mdata, x="band", y="base_accuracy", ax=ax, palette=BAND_COLORS, order=BANDS
    )
    sns.stripplot(
        data=mdata,
        x="band",
        y="base_accuracy",
        ax=ax,
        color="black",
        alpha=0.6,
        size=5,
        order=BANDS,
    )
    ax.set_title(model, fontsize=13)
    ax.set_xticklabels(get_band_labels(), rotation=30, ha="right")
    ax.set_xlabel("Frequency Band")
    if i == 0:
        ax.set_ylabel("Base Model Accuracy")
    else:
        ax.set_ylabel("")
fig.suptitle("Base Model Accuracy by Frequency Band", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "01_base_model_accuracy.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/01_base_model_accuracy.png


## 4. Circuit Structure (Minimality)

How large are the discovered circuits relative to the full model graph?

In [6]:
size_summary = (
    df_circuit.groupby(["model", "band"], observed=True)
    .agg(
        n_edges_mean=("n_edges", "mean"),
        n_edges_std=("n_edges", "std"),
        size_fraction_mean=("size_fraction", "mean"),
        size_fraction_std=("size_fraction", "std"),
        total_edges=("total_edges", "first"),
    )
    .reset_index()
)
print(size_summary.to_string(index=False))
size_summary.to_csv(ANALYSIS_DIR / "circuit_size_stats.csv", index=False)
print(f"\nSaved: circuit_size_stats.csv")

      model      band  n_edges_mean  n_edges_std  size_fraction_mean  size_fraction_std  total_edges
 pythia-70m       low    389.000000     8.185353            0.293807           0.006182         1324
 pythia-70m    medium    397.666667     7.371115            0.300352           0.005567         1324
 pythia-70m      high    416.666667    13.203535            0.314703           0.009972         1324
 pythia-70m very_high    419.333333     4.041452            0.316717           0.003052         1324
 pythia-70m   control    423.666667    12.662280            0.319990           0.009564         1324
pythia-160m       low   1451.333333    38.837267            0.126566           0.003387        11467
pythia-160m    medium   1478.000000    33.600595            0.128892           0.002930        11467
pythia-160m      high   1399.666667    48.686069            0.122060           0.004246        11467
pythia-160m very_high   1331.333333    26.025628            0.116101           0.002270    

### VIZ 02: Circuit Size (Model x Band Heatmap)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

plot_metric_heatmap(
    df_circuit,
    "size_fraction",
    "Edge Fraction (Circuit / Total)",
    ax=axes[0],
    cmap="YlOrRd",
    fmt=".3f",
)

plot_metric_heatmap(
    df_circuit,
    "n_edges",
    "Circuit Edges (absolute count)",
    ax=axes[1],
    cmap="YlOrRd",
    fmt=".0f",
)

fig.suptitle("Circuit Minimality", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "02_circuit_size_heatmap.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/02_circuit_size_heatmap.png


## 5. Same-Band Performance (Faithfulness)

How well does the circuit reproduce full-model performance on its own band?

In [8]:
faith_summary = (
    df_circuit.groupby(["model", "band"], observed=True)
    .agg(
        circuit_acc_mean=("circuit_accuracy", "mean"),
        circuit_acc_std=("circuit_accuracy", "std"),
        circuit_top5_mean=("circuit_top5_accuracy", "mean"),
        circuit_top5_std=("circuit_top5_accuracy", "std"),
        circuit_kl_mean=("circuit_kl_div", "mean"),
        circuit_kl_std=("circuit_kl_div", "std"),
        retention_mean=("retention_ratio", "mean"),
        retention_std=("retention_ratio", "std"),
    )
    .reset_index()
)
print(faith_summary.to_string(index=False))
faith_summary.to_csv(ANALYSIS_DIR / "faithfulness_stats.csv", index=False)
print(f"\nSaved: faithfulness_stats.csv")

      model      band  circuit_acc_mean  circuit_acc_std  circuit_top5_mean  circuit_top5_std  circuit_kl_mean  circuit_kl_std  retention_mean  retention_std
 pythia-70m       low          0.242963         0.059018           0.392593          0.056627         0.229412        0.017591        0.802972       0.065225
 pythia-70m    medium          0.348148         0.035924           0.466667          0.035277         0.241756        0.010827        0.869060       0.040471
 pythia-70m      high          0.416296         0.045614           0.585185          0.056627         0.271722        0.042201        0.829583       0.073516
 pythia-70m very_high          0.549630         0.037007           0.720000          0.034712         0.255592        0.016166        0.820326       0.017721
 pythia-70m   control          0.525926         0.015608           0.712593          0.047523         0.257002        0.021475        0.884624       0.044935
pythia-160m       low          0.890370         0.01

### VIZ 03: Same-Band Accuracy Heatmap

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

plot_metric_heatmap(
    df_circuit,
    "circuit_accuracy",
    "Circuit Accuracy (same-band)",
    ax=axes[0],
    cmap="RdYlGn",
    fmt=".3f",
    vmin=0,
    vmax=1,
)

plot_metric_heatmap(
    df_circuit,
    "retention_ratio",
    "Retention Ratio (circuit / base)",
    ax=axes[1],
    cmap="RdYlGn",
    fmt=".3f",
    vmin=0,
    vmax=1,
)

fig.suptitle("Circuit Faithfulness", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "03_same_band_accuracy_heatmap.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/03_same_band_accuracy_heatmap.png


### VIZ 04: Retention Ratio by Band

In [10]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=df_circuit,
    x="band",
    y="retention_ratio",
    hue="model",
    ax=ax,
    palette=MODEL_COLORS,
    order=BANDS,
    errorbar="sd",
    capsize=0.05,
)
ax.set_xticklabels(get_band_labels())
ax.set_xlabel("Frequency Band")
ax.set_ylabel("Retention Ratio (Circuit Acc / Base Acc)")
ax.set_title("Retention Ratio by Frequency Band and Model", fontsize=14)
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
fig.tight_layout()
save_figure(fig, "04_retention_ratio_by_band.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/04_retention_ratio_by_band.png


## 6. Cross-Band Generalization

How well does a circuit trained on one band perform on other bands?

In [11]:
transfer_matrices = {}
for model in MODELS:
    transfer_matrices[model] = compute_transfer_matrix(df_transfer, model)
    print(f"\n{model} Transfer Matrix (accuracy):")
    print(transfer_matrices[model].round(3).to_string())


pythia-70m Transfer Matrix (accuracy):
test_band     low  medium   high  very_high  control
train_band                                          
low         0.243   0.323  0.359      0.468    0.433
medium      0.252   0.348  0.388      0.511    0.453
high        0.270   0.345  0.416      0.530    0.498
very_high   0.231   0.327  0.394      0.550    0.489
control     0.262   0.356  0.422      0.569    0.526

pythia-160m Transfer Matrix (accuracy):
test_band     low  medium   high  very_high  control
train_band                                          
low         0.890   0.920  0.919      0.914    0.905
medium      0.844   0.911  0.927      0.913    0.920
high        0.834   0.892  0.923      0.923    0.913
very_high   0.833   0.887  0.929      0.951    0.930
control     0.837   0.899  0.920      0.941    0.936

pythia-410m Transfer Matrix (accuracy):
test_band     low  medium   high  very_high  control
train_band                                          
low         0.947   0.976  0.9

### VIZ 05: Cross-Band Transfer Matrices

In [12]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(7 * len(MODELS), 6))
for i, model in enumerate(MODELS):
    plot_transfer_heatmap(
        transfer_matrices[model],
        title=model,
        ax=axes[i],
        vmin=0,
        vmax=1,
    )
fig.suptitle("Cross-Band Transfer Matrices (Circuit Accuracy)", fontsize=16, y=1.02)
fig.tight_layout()
save_figure(fig, "05_cross_band_transfer_matrices.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/05_cross_band_transfer_matrices.png


In [13]:
gap_dfs = []
for model in MODELS:
    gap_dfs.append(compute_generalization_gap(df_transfer, model))
df_gaps = pd.concat(gap_dfs, ignore_index=True)

gap_summary = (
    df_gaps.groupby(["model", "train_band"])
    .agg(
        same_band_mean=("same_band_metric", "mean"),
        cross_band_mean_mean=("cross_band_mean", "mean"),
        gap_mean=("gap", "mean"),
        gap_std=("gap", "std"),
        transfer_ratio_mean=("transfer_ratio", "mean"),
    )
    .reset_index()
)
print("Generalization Gap Summary:")
print(gap_summary.to_string(index=False))
gap_summary.to_csv(ANALYSIS_DIR / "generalization_gap_stats.csv", index=False)
df_gaps.to_csv(ANALYSIS_DIR / "generalization_gap_per_draw.csv", index=False)
print(f"\nSaved: generalization_gap_stats.csv, generalization_gap_per_draw.csv")

Generalization Gap Summary:
      model train_band  same_band_mean  cross_band_mean_mean  gap_mean  gap_std  transfer_ratio_mean
pythia-1.4b    control        0.924444              0.856296  0.068148 0.021266             0.926437
pythia-1.4b       high        0.894815              0.845926  0.048889 0.027307             0.945207
pythia-1.4b        low        0.798519              0.912963 -0.114444 0.030082             1.144117
pythia-1.4b     medium        0.859259              0.891481 -0.032222 0.019277             1.037675
pythia-1.4b  very_high        0.918519              0.775185  0.143333 0.017105             0.844021
pythia-160m    control        0.936296              0.899259  0.037037 0.023156             0.960628
pythia-160m       high        0.922963              0.890370  0.032593 0.015167             0.964780
pythia-160m        low        0.890370              0.914444 -0.024074 0.012636             1.027077
pythia-160m     medium        0.911111              0.901111  0

## 7. Asymmetric Transfer

Do low-frequency circuits generalize to high-frequency data better than vice versa (or the opposite)?

- **LF bands**: low, medium
- **HF bands**: high, very_high

In [14]:
asymm_results = []
for model in MODELS:
    result = compute_asymmetric_transfer(df_transfer, model)
    result["model"] = model
    asymm_results.append(result)
    print(
        f"{model}: LF->HF={result['lf_to_hf_mean']:.3f}, "
        f"HF->LF={result['hf_to_lf_mean']:.3f}, "
        f"Asymmetry(LF->HF - HF->LF)={result['asymmetry']:.3f}"
    )

asymm_df = pd.DataFrame(
    [
        {
            "model": r["model"],
            "lf_to_hf": r["lf_to_hf_mean"],
            "hf_to_lf": r["hf_to_lf_mean"],
            "asymmetry": r["asymmetry"],
            "asymmetry_ratio": r["asymmetry_ratio"],
        }
        for r in asymm_results
    ]
)
asymm_df.to_csv(ANALYSIS_DIR / "asymmetry_summary.csv", index=False)
print(f"\nSaved: asymmetry_summary.csv")

pythia-70m: LF->HF=0.431, HF->LF=0.293, Asymmetry(LF->HF - HF->LF)=0.138
pythia-160m: LF->HF=0.918, HF->LF=0.861, Asymmetry(LF->HF - HF->LF)=0.057
pythia-410m: LF->HF=0.965, HF->LF=0.917, Asymmetry(LF->HF - HF->LF)=0.047
pythia-1b: LF->HF=0.925, HF->LF=0.834, Asymmetry(LF->HF - HF->LF)=0.091
pythia-1.4b: LF->HF=0.921, HF->LF=0.717, Asymmetry(LF->HF - HF->LF)=0.204

Saved: asymmetry_summary.csv


### VIZ 06: Asymmetric Transfer

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: LF->HF vs HF->LF per model
ax = axes[0]
x = np.arange(len(MODELS))
width = 0.35
bars1 = ax.bar(
    x - width / 2,
    asymm_df["lf_to_hf"],
    width,
    label="LF -> HF",
    color="#d62728",
    alpha=0.8,
)
bars2 = ax.bar(
    x + width / 2,
    asymm_df["hf_to_lf"],
    width,
    label="HF -> LF",
    color="#1f77b4",
    alpha=0.8,
)
ax.set_xticks(x)
ax.set_xticklabels(MODELS)
ax.set_ylabel("Transfer Accuracy")
ax.set_title("Cross-Frequency Transfer Direction")
ax.legend()
ax.bar_label(bars1, fmt="%.3f", padding=2, fontsize=9)
ax.bar_label(bars2, fmt="%.3f", padding=2, fontsize=9)

# Right: Asymmetry magnitude
ax = axes[1]
colors = ["#2ca02c" if a > 0 else "#d62728" for a in asymm_df["asymmetry"]]
bars = ax.bar(MODELS, asymm_df["asymmetry"], color=colors, alpha=0.8)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_ylabel("Asymmetry (LF->HF - HF->LF)")
ax.set_title("Transfer Asymmetry by Model")
ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=9)

fig.suptitle("Asymmetric Transfer Analysis", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "06_asymmetric_transfer.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/06_asymmetric_transfer.png


In [16]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(7 * len(MODELS), 6))
for i, model in enumerate(MODELS):
    asymm_matrix = compute_pairwise_asymmetry_matrix(df_transfer, model)
    ax = axes[i]
    labels = get_band_labels(FREQUENCY_BANDS)
    sns.heatmap(
        asymm_matrix,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        center=0,
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
        xticklabels=labels,
        yticklabels=labels,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(f"{model}", fontsize=13)
    ax.set_xlabel("Test Band")
    ax.set_ylabel("Train Band")
fig.suptitle(
    "Pairwise Transfer Asymmetry (row->col minus col->row)", fontsize=15, y=1.02
)
fig.tight_layout()
save_figure(fig, "06b_pairwise_asymmetry.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/06b_pairwise_asymmetry.png


In [17]:
# VIZ 06c: Pairwise Asymmetry vs Frequency Distance
# Compute pairwise asymmetry for all C(4,2)=6 directed pairs among frequency bands
pw_results = []
for i, band_a in enumerate(FREQUENCY_BANDS):
    for j, band_b in enumerate(FREQUENCY_BANDS):
        if i >= j:
            continue
        freq_dist = abs(FREQUENCY_RANK[band_a] - FREQUENCY_RANK[band_b])
        a_to_b = df_transfer[
            (df_transfer["train_band"] == band_a) & (df_transfer["test_band"] == band_b)
        ]["circuit_accuracy"].mean()
        b_to_a = df_transfer[
            (df_transfer["train_band"] == band_b) & (df_transfer["test_band"] == band_a)
        ]["circuit_accuracy"].mean()
        pw_results.append(
            {
                "band_a": band_a,
                "band_b": band_b,
                "a_to_b": a_to_b,
                "b_to_a": b_to_a,
                "asymmetry": a_to_b - b_to_a,
                "abs_asymmetry": abs(a_to_b - b_to_a),
                "freq_distance": freq_dist,
            }
        )
df_pw = pd.DataFrame(pw_results)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Full 4x4 directed asymmetry matrix
ax = axes[0]
asym_matrix = np.zeros((len(FREQUENCY_BANDS), len(FREQUENCY_BANDS)))
for _, row in df_pw.iterrows():
    i = FREQUENCY_BANDS.index(row["band_a"])
    j = FREQUENCY_BANDS.index(row["band_b"])
    asym_matrix[i, j] = row["asymmetry"]
    asym_matrix[j, i] = -row["asymmetry"]

band_labels = [BAND_NAMES[b] for b in FREQUENCY_BANDS]
vmax = max(abs(asym_matrix.min()), abs(asym_matrix.max()))
im = ax.imshow(asym_matrix, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="equal")
ax.set_xticks(range(len(band_labels)))
ax.set_xticklabels(band_labels, rotation=45, ha="right")
ax.set_yticks(range(len(band_labels)))
ax.set_yticklabels(band_labels)
ax.set_xlabel("Test Band")
ax.set_ylabel("Train Band")
ax.set_title(
    "Directed Transfer Asymmetry\n(train_row -> test_col) - (test_col -> train_row)"
)
for i in range(len(band_labels)):
    for j in range(len(band_labels)):
        if i != j:
            ax.text(
                j,
                i,
                f"{asym_matrix[i, j]:+.3f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if abs(asym_matrix[i, j]) > vmax * 0.6 else "black",
            )
plt.colorbar(im, ax=ax, shrink=0.8)
ax.grid(False)

# Right: Asymmetry magnitude vs frequency distance
ax = axes[1]
ax.scatter(
    df_pw["freq_distance"],
    df_pw["abs_asymmetry"],
    s=100,
    c="steelblue",
    edgecolors="black",
    zorder=5,
)
for _, row in df_pw.iterrows():
    ax.annotate(
        f"{row['band_a'][:3]}-{row['band_b'][:3]}",
        (row["freq_distance"], row["abs_asymmetry"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=8,
    )

# Add trend line
from scipy.stats import spearmanr

rho, p_val = spearmanr(df_pw["freq_distance"], df_pw["abs_asymmetry"])
z = np.polyfit(df_pw["freq_distance"], df_pw["abs_asymmetry"], 1)
poly = np.poly1d(z)
x_range = np.linspace(
    df_pw["freq_distance"].min() - 0.1, df_pw["freq_distance"].max() + 0.1, 50
)
ax.plot(x_range, poly(x_range), "--", color="coral", alpha=0.7)
ax.set_xlabel("Frequency Rank Distance")
ax.set_ylabel("|Asymmetry Magnitude|")
ax.set_title(
    f"Asymmetry vs Frequency Distance\n(Spearman rho={rho:.3f}, p={p_val:.4f})"
)

fig.suptitle("Pairwise Band Asymmetry Analysis", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "06c_pairwise_asymmetry_distance.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/06c_pairwise_asymmetry_distance.png


## 8. Ablation Analysis (Completeness)

Are the discovered circuit edges necessary? (Removing them should destroy performance.)

In [18]:
comp_summary = (
    df_circuit.groupby(["model", "band"], observed=True)
    .agg(
        ablation_acc_mean=("ablation_accuracy", "mean"),
        ablation_acc_std=("ablation_accuracy", "std"),
        completeness_mean=("completeness", "mean"),
        completeness_std=("completeness", "std"),
    )
    .reset_index()
)
print(comp_summary.to_string(index=False))
comp_summary.to_csv(ANALYSIS_DIR / "completeness_stats.csv", index=False)
print(f"\nSaved: completeness_stats.csv")

      model      band  ablation_acc_mean  ablation_acc_std  completeness_mean  completeness_std
 pythia-70m       low           0.000000          0.000000           1.000000          0.000000
 pythia-70m    medium           0.000000          0.000000           1.000000          0.000000
 pythia-70m      high           0.000000          0.000000           1.000000          0.000000
 pythia-70m very_high           0.000000          0.000000           1.000000          0.000000
 pythia-70m   control           0.000000          0.000000           1.000000          0.000000
pythia-160m       low           0.000000          0.000000           1.000000          0.000000
pythia-160m    medium           0.000000          0.000000           1.000000          0.000000
pythia-160m      high           0.001481          0.002566           0.998519          0.002566
pythia-160m very_high           0.000000          0.000000           1.000000          0.000000
pythia-160m   control           0.000000

### VIZ 07: Three-Way Comparison (Base vs Circuit vs Ablation)

In [19]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 6), sharey=True)
for i, model in enumerate(MODELS):
    ax = axes[i]
    mdata = df_circuit[df_circuit["model"] == model].copy()
    melt = mdata.melt(
        id_vars=["band"],
        value_vars=["base_accuracy", "circuit_accuracy", "ablation_accuracy"],
        var_name="type",
        value_name="accuracy",
    )
    melt["type"] = melt["type"].map(
        {
            "base_accuracy": "Base",
            "circuit_accuracy": "Circuit",
            "ablation_accuracy": "Ablation",
        }
    )
    sns.barplot(
        data=melt,
        x="band",
        y="accuracy",
        hue="type",
        ax=ax,
        order=BANDS,
        hue_order=["Base", "Circuit", "Ablation"],
        palette={"Base": "#2ca02c", "Circuit": "#1f77b4", "Ablation": "#d62728"},
        errorbar="sd",
        capsize=0.03,
    )
    ax.set_title(model, fontsize=13)
    ax.set_xticklabels(get_band_labels(), rotation=30, ha="right")
    ax.set_xlabel("Frequency Band")
    if i == 0:
        ax.set_ylabel("Accuracy")
    else:
        ax.set_ylabel("")
    if i == len(MODELS) - 1:
        ax.legend(title="Evaluation", bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.get_legend().remove()

fig.suptitle("Three-Way Comparison: Base vs Circuit vs Ablation", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "07_three_way_comparison.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/07_three_way_comparison.png


## 9. Model Scaling Analysis

How do circuit properties change across model sizes (70M -> 160M -> 410M -> 1B)?

In [20]:
df_scaling = compute_model_scaling_df(df_circuit)
scaling_summary = (
    df_scaling.groupby(["model", "model_size"], observed=True)
    .agg(
        base_acc=("base_accuracy", "mean"),
        circuit_acc=("circuit_accuracy", "mean"),
        circuit_acc_std=("circuit_accuracy", "std"),
        size_fraction=("size_fraction", "mean"),
        retention=("retention_ratio", "mean"),
        kl_div=("circuit_kl_div", "mean"),
        n_edges=("n_edges", "mean"),
    )
    .reset_index()
)
print(scaling_summary.to_string(index=False))
scaling_summary.to_csv(ANALYSIS_DIR / "scaling_summary.csv", index=False)
print(f"\nSaved: scaling_summary.csv")

      model model_size  base_acc  circuit_acc  circuit_acc_std  size_fraction  retention   kl_div     n_edges
 pythia-70m         70  0.493630     0.416593         0.122588       0.309114   0.841313 0.251097  409.266667
pythia-160m        160  0.958519     0.922370         0.024219       0.122659   0.962267 0.292350 1406.533333
pythia-410m        410  0.988444     0.964148         0.013383       0.044442   0.975425 0.320098 3581.200000
  pythia-1b       1000  0.987259     0.930667         0.022525       0.091404   0.942633 0.523258  914.866667
pythia-1.4b       1400  0.979259     0.879111         0.050176       0.028256   0.897346 0.558626 2276.866667

Saved: scaling_summary.csv


### VIZ 08: Model Scaling Trends

In [21]:
fig = plot_scaling_panel(
    df_circuit,
    metrics=["circuit_accuracy", "size_fraction", "retention_ratio", "circuit_kl_div"],
    titles=["Circuit Accuracy", "Edge Fraction", "Retention Ratio", "KL Divergence"],
    ylabels=["Accuracy", "Fraction", "Ratio", "KL Div"],
    suptitle="Circuit Properties vs Model Size",
)
save_figure(fig, "08_model_scaling.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/08_model_scaling.png


### VIZ 09: Accuracy vs Sparsity

In [22]:
fig, ax = plt.subplots(figsize=(12, 8))
markers = {
    "pythia-70m": "o",
    "pythia-160m": "s",
    "pythia-410m": "^",
    "pythia-1b": "D",
    "pythia-1.4b": "P",
}
for model in MODELS:
    mdata = df_circuit[df_circuit["model"] == model]
    for band in BANDS:
        bdata = mdata[mdata["band"] == band]
        ax.scatter(
            bdata["size_fraction"],
            bdata["circuit_accuracy"],
            color=BAND_COLORS[band],
            marker=markers[model],
            s=80,
            alpha=0.7,
            edgecolors="black",
            linewidth=0.5,
        )

from matplotlib.lines import Line2D

band_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=BAND_COLORS[b],
        markersize=10,
        label=BAND_NAMES[b],
    )
    for b in BANDS
]
model_handles = [
    Line2D(
        [0],
        [0],
        marker=markers[m],
        color="w",
        markerfacecolor="gray",
        markersize=10,
        label=m,
    )
    for m in MODELS
]
legend1 = ax.legend(handles=band_handles, title="Band", loc="upper left", fontsize=9)
ax.add_artist(legend1)
ax.legend(handles=model_handles, title="Model", loc="lower right", fontsize=9)

ax.set_xlabel("Edge Fraction (lower = sparser)")
ax.set_ylabel("Circuit Accuracy (higher = better)")
ax.set_title("Accuracy vs Sparsity Trade-off", fontsize=14)
fig.tight_layout()
save_figure(fig, "09_accuracy_vs_sparsity.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/09_accuracy_vs_sparsity.png


### VIZ 10: KL Divergence Analysis

In [23]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sns.boxplot(
    data=df_circuit,
    x="band",
    y="circuit_kl_div",
    hue="model",
    ax=ax,
    palette=MODEL_COLORS,
    order=BANDS,
)
ax.set_xticklabels(get_band_labels(), rotation=30, ha="right")
ax.set_xlabel("Frequency Band")
ax.set_ylabel("KL Divergence (circuit vs base)")
ax.set_title("Circuit KL Divergence")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

plot_metric_heatmap(
    df_circuit,
    "circuit_kl_div",
    "KL Divergence (model x band)",
    ax=axes[1],
    cmap="YlOrRd",
    fmt=".3f",
)

fig.suptitle("KL Divergence Analysis", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "10_kl_divergence_analysis.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/10_kl_divergence_analysis.png


## 10. Aggregated Summary & Export

In [24]:
master = (
    df_circuit.groupby(["model", "band"], observed=True)
    .agg(
        base_acc=("base_accuracy", "mean"),
        circuit_acc=("circuit_accuracy", "mean"),
        circuit_acc_std=("circuit_accuracy", "std"),
        circuit_top5=("circuit_top5_accuracy", "mean"),
        kl_div=("circuit_kl_div", "mean"),
        ablation_acc=("ablation_accuracy", "mean"),
        completeness=("completeness", "mean"),
        retention=("retention_ratio", "mean"),
        size_frac=("size_fraction", "mean"),
        n_edges=("n_edges", "mean"),
    )
    .reset_index()
)
print("MASTER SUMMARY")
print("=" * 120)
print(master.to_string(index=False))
master.to_csv(ANALYSIS_DIR / "master_summary.csv", index=False)
print(f"\nSaved: master_summary.csv")

MASTER SUMMARY
      model      band  base_acc  circuit_acc  circuit_acc_std  circuit_top5   kl_div  ablation_acc  completeness  retention  size_frac     n_edges
 pythia-70m       low  0.300741     0.242963         0.059018      0.392593 0.229412      0.000000      1.000000   0.802972   0.293807  389.000000
 pythia-70m    medium  0.400000     0.348148         0.035924      0.466667 0.241756      0.000000      1.000000   0.869060   0.300352  397.666667
 pythia-70m      high  0.502222     0.416296         0.045614      0.585185 0.271722      0.000000      1.000000   0.829583   0.314703  416.666667
 pythia-70m very_high  0.669630     0.549630         0.037007      0.720000 0.255592      0.000000      1.000000   0.820326   0.316717  419.333333
 pythia-70m   control  0.595556     0.525926         0.015608      0.712593 0.257002      0.000000      1.000000   0.884624   0.319990  423.666667
pythia-160m       low  0.928889     0.890370         0.012830      0.970370 0.305093      0.000000     

In [25]:
df_circuit.to_csv(ANALYSIS_DIR / "full_circuit_data.csv", index=False)
df_transfer.to_csv(ANALYSIS_DIR / "full_transfer_data.csv", index=False)
print(f"Saved: full_circuit_data.csv ({len(df_circuit)} rows)")
print(f"Saved: full_transfer_data.csv ({len(df_transfer)} rows)")

Saved: full_circuit_data.csv (75 rows)
Saved: full_transfer_data.csv (375 rows)


In [26]:
print("\n" + "=" * 80)
print("EXPORTED FILES")
print("=" * 80)
print("\nCSV files:")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}")
print("\nVisualizations:")
for f in sorted(VIZ_DIR.glob("*.png")):
    print(f"  {f.name}")


EXPORTED FILES

CSV files:
  all_statistical_tests.csv
  asymmetry_per_draw.csv
  asymmetry_summary.csv
  base_model_stats.csv
  bimodality_results.csv
  circuit_size_stats.csv
  completeness_stats.csv
  failure_analysis_summary.csv
  faithfulness_stats.csv
  full_circuit_data.csv
  full_transfer_data.csv
  generalization_gap_per_draw.csv
  generalization_gap_stats.csv
  master_summary.csv
  pairwise_band_asymmetry.csv
  per_example_robustness.csv
  random_baseline_summary.csv
  scaling_summary.csv
  summary_by_domain.csv
  variance_decomposition.csv

Visualizations:
  01_base_model_accuracy.png
  02_circuit_size_heatmap.png
  03_same_band_accuracy_heatmap.png
  04_retention_ratio_by_band.png
  05_cross_band_transfer_matrices.png
  06_asymmetric_transfer.png
  06b_pairwise_asymmetry.png
  06c_pairwise_asymmetry_distance.png
  07_three_way_comparison.png
  08_model_scaling.png
  09_accuracy_vs_sparsity.png
  10_kl_divergence_analysis.png
  11_effect_sizes_by_domain.png
  12_significanc

### VIZ 17: Random Baseline Comparison

For each circuit, compare the real ACDC-discovered accuracy against K=100 random edge sets of the same size. Real circuits should dramatically outperform random baselines.

**Prerequisite**: Run `lsc_random_baseline.py` first.

In [27]:
# VIZ 17: Random Baseline: real vs random circuit accuracy
rb_path = RANDOM_BASELINE_RESULTS
if rb_path.exists():
    with open(rb_path) as f:
        rb_data = json.load(f)
    df_rb = pd.DataFrame(rb_data["results"])
    print(f"Loaded {len(df_rb)} random baseline results (K={rb_data.get('K', '?')})")

    n_models = len(MODELS)

    ncols = 3

    nrows = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
    axes_flat = axes.flatten()

    for idx, model in enumerate(MODELS):
        ax = axes_flat[idx]
        mdata = df_rb[df_rb["model"] == model].sort_values(["band", "draw"])

        if len(mdata) == 0:
            ax.set_title(f"{model} (no data)")
            continue

        x_labels = [
            f"{row['band'][:3]}\n{row['draw'][-1]}" for _, row in mdata.iterrows()
        ]
        x = np.arange(len(mdata))

        # Random baseline as bar (mean +/- std)
        ax.bar(
            x,
            mdata["mean_random_accuracy"],
            width=0.6,
            color="lightcoral",
            alpha=0.6,
            label="Random (mean)",
        )
        ax.errorbar(
            x,
            mdata["mean_random_accuracy"],
            yerr=mdata["std_random_accuracy"],
            fmt="none",
            color="darkred",
            capsize=3,
            alpha=0.7,
        )

        # Real accuracy as stars
        ax.scatter(
            x,
            mdata["real_accuracy"],
            marker="*",
            s=200,
            c=[BAND_COLORS.get(b, "gray") for b in mdata["band"]],
            edgecolors="black",
            linewidths=0.5,
            zorder=10,
            label="Real circuit",
        )

        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, fontsize=7)
        ax.set_xlabel("Band / Draw")
        ax.set_ylabel("Accuracy")
        ax.set_title(f"{model}", fontsize=13)
        ax.legend(fontsize=9, loc="upper left")
        ax.grid(axis="y", alpha=0.3)

    for idx in range(n_models, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle("ACDC Circuit vs Random Edge Sets (same size)", fontsize=15, y=1.02)
    fig.tight_layout()
    save_figure(fig, "17_random_baseline.png")
else:
    print(f"Random baseline results not found at {rb_path}")
    print("Run lsc_random_baseline.py first, then re-run this cell.")

Loaded 60 random baseline results (K=100)


  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/17_random_baseline.png


### VIZ 18: Control as Average Circuit

Does the control circuit behave as a "jack of all trades, master of none"?
- Each line shows a train-band circuit's accuracy when tested on all 5 bands
- Control (dashed) should be flat and near-average, while frequency-specific circuits peak on their own band

In [28]:
# VIZ 18: Control as Average Circuit: transfer profiles per model
n_models = len(MODELS)

ncols = 3

nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

test_band_order = BANDS  # low, medium, high, very_high, control
test_labels = [BAND_NAMES[b] for b in test_band_order]

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    mdata = df_transfer[df_transfer["model"] == model]

    for train_band in BANDS:
        tdata = mdata[mdata["train_band"] == train_band]
        # Mean accuracy per test band (averaged over draws)
        profile = tdata.groupby("test_band")["circuit_accuracy"].mean()
        profile = profile.reindex(test_band_order)

        is_control = train_band == "control"
        ax.plot(
            range(len(test_band_order)),
            profile.values,
            color=BAND_COLORS[train_band],
            linewidth=3.0 if is_control else 1.5,
            linestyle="--" if is_control else "-",
            marker="o" if is_control else "s",
            markersize=8 if is_control else 5,
            alpha=1.0 if is_control else 0.7,
            label=BAND_NAMES[train_band],
            zorder=10 if is_control else 5,
        )

    ax.set_xticks(range(len(test_band_order)))
    ax.set_xticklabels(test_labels, rotation=30, ha="right")
    ax.set_xlabel("Test Band")
    ax.set_ylabel("Circuit Accuracy")
    ax.set_title(f"{model}", fontsize=13)
    ax.legend(title="Train Band", fontsize=8, loc="lower right")
    ax.grid(True, alpha=0.3)

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)


fig.suptitle(
    "Control Circuit Transfer Profile vs Frequency-Specific Circuits",
    fontsize=15,
    y=1.02,
)
fig.tight_layout()
save_figure(fig, "18_control_average_circuit.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/18_control_average_circuit.png


## Key Observations

In [29]:
print("=" * 80)
print("KEY OBSERVATIONS")
print("=" * 80)

for model in MODELS:
    mdata = df_circuit[df_circuit["model"] == model]
    low_base = mdata[mdata["band"] == "low"]["base_accuracy"].mean()
    vh_base = mdata[mdata["band"] == "very_high"]["base_accuracy"].mean()
    print(f"\n{model}:")
    print(
        f"  Base acc: low={low_base:.3f}, very_high={vh_base:.3f} (ratio={vh_base / low_base:.2f}x)"
    )
    low_circ = mdata[mdata["band"] == "low"]["circuit_accuracy"].mean()
    vh_circ = mdata[mdata["band"] == "very_high"]["circuit_accuracy"].mean()
    print(f"  Circuit acc: low={low_circ:.3f}, very_high={vh_circ:.3f}")
    low_ret = mdata[mdata["band"] == "low"]["retention_ratio"].mean()
    vh_ret = mdata[mdata["band"] == "very_high"]["retention_ratio"].mean()
    print(f"  Retention: low={low_ret:.3f}, very_high={vh_ret:.3f}")

print("\n\nAsymmetric Transfer Summary:")
for _, row in asymm_df.iterrows():
    direction = "LF->HF better" if row["asymmetry"] > 0 else "HF->LF better"
    print(f"  {row['model']}: asymmetry={row['asymmetry']:.3f} ({direction})")

mean_gap = df_gaps.groupby("model")["gap"].mean()
print("\nMean Generalization Gap (same-band - cross-band):")
for model, gap in mean_gap.items():
    print(f"  {model}: {gap:.4f}")

KEY OBSERVATIONS

pythia-70m:
  Base acc: low=0.301, very_high=0.670 (ratio=2.23x)
  Circuit acc: low=0.243, very_high=0.550
  Retention: low=0.803, very_high=0.820

pythia-160m:
  Base acc: low=0.929, very_high=0.969 (ratio=1.04x)
  Circuit acc: low=0.890, very_high=0.951
  Retention: low=0.959, very_high=0.982

pythia-410m:
  Base acc: low=0.982, very_high=0.991 (ratio=1.01x)
  Circuit acc: low=0.947, very_high=0.963
  Retention: low=0.964, very_high=0.972

pythia-1b:
  Base acc: low=0.976, very_high=0.991 (ratio=1.02x)
  Circuit acc: low=0.905, very_high=0.936
  Retention: low=0.927, very_high=0.945

pythia-1.4b:
  Base acc: low=0.959, very_high=0.994 (ratio=1.04x)
  Circuit acc: low=0.799, very_high=0.919
  Retention: low=0.833, very_high=0.924


Asymmetric Transfer Summary:
  pythia-70m: asymmetry=0.138 (LF->HF better)
  pythia-160m: asymmetry=0.057 (LF->HF better)
  pythia-410m: asymmetry=0.047 (LF->HF better)
  pythia-1b: asymmetry=0.091 (LF->HF better)
  pythia-1.4b: asymmetry=